# YZV311E Data Mining Course Project

## Data Merging

### Links to the datasets:

https://www.kaggle.com/datasets/venky73/spam-mails-dataset

https://www.kaggle.com/datasets/shantanudhakadd/email-spam-detection-dataset-classification

https://www.kaggle.com/datasets/nitishabharathi/email-spam-dataset

https://www.kaggle.com/datasets/wcukierski/enron-email-dataset

https://www.kaggle.com/datasets/subhajournal/phishingemails

https://www.kaggle.com/datasets/jackksoncsie/spam-email-dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PATH = "datasets/"

In [ ]:
data1 = pd.read_csv(PATH + "spam.csv", encoding='latin-1')

In [ ]:
data1.count()

In [ ]:
# lets check if Unnamed: 2, 3 and 4 consist any spam messages
data1[data1['v1'] == 'spam'].count()

Now, Unnamed 2, 3 and 4 are replies to the mails people responded to. It is most likely that it will not consist spam data (and as we saw, this might fall into the category of outliers. So, we will remove these columns.)

In [ ]:
data1.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1, inplace=True)
# With this operation, we will have 2 clean columns since the dropped columns also includede texts like .;-):-D" etc.

In [ ]:
data1.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data1['v1'] = le.fit_transform(data1['v1'])

data1.head()

In [ ]:
data1.rename(columns={'v1': 'label', 'v2': 'text'}, inplace=True)

In [ ]:
data1.head()

### 1 is spam, 0 is not spam

In [ ]:
data2 = pd.read_csv(PATH + "spam_ham_dataset.csv", encoding='latin-1')

In [ ]:
data2.head()

In [ ]:
data2.drop(["Unnamed: 0", "label"], axis=1, inplace=True)

In [ ]:
data2["text"]

While merging, we will create a subject column and a body column, we understand that not every dataset included the subjects but seperating them into a different dataset will help us analyse the data better.

In [ ]:
# find keyword "Subject:" and count it 
data2[data2["text"].str.contains("Subject:")].count()

In [ ]:
# whole data row count
data2.count()

So all text data includes a subject and a body column.

In [ ]:
# what we know is in text column the subject ends right after \r and the body starts after \n
# so we will split the text column into 2 columns

data2["subject"] = data2["text"].str.split("\r").str[0]
data2["body"] = data2["text"].str.split("\n").str[1]

In [ ]:
data2.head()

In [ ]:
data2.drop(["text"], axis=1, inplace=True)
data2.rename(columns={'label_num': 'label', "body": "text"}, inplace=True)

In [ ]:
data2.head()

In [ ]:
data3 = pd.read_csv(PATH + "Phishing_Email.csv", encoding='latin-1')

In [ ]:
data3.head()

In [ ]:
data3['label'] = data3['Email Type'].apply(lambda x: ["Safe Email", "Phishing Email"].index(x))
# Source: https://stackoverflow.com/questions/38749305/labelencoder-order-of-fit-for-a-pandas-df 
# label encoder was encoding safe email as 1 and we had no way of having an order for the labels

In [ ]:
data3.head()

In [ ]:
data3.drop(columns=["Email Type", "Unnamed: 0"], inplace=True)

In [ ]:
data3.rename(columns={'Email Text': 'text'}, inplace=True)

In [ ]:
data3.head()

In [ ]:
data4 = pd.read_csv(PATH + "emails.csv", encoding='latin-1')

In [ ]:
data4.head()

In [ ]:
# for this specific dataset, the subject ends right after "  " which is two spaces. we will split the text column into 2 columns
data4["subject"] = data4["text"].str.split("  ").str[0]
data4["text"] = data4["text"].str.split("  ").str[1]

In [ ]:
data4.head()

In [ ]:
data5 = pd.read_csv(PATH + "spam_assassin/completeSpamAssassin.csv", encoding='latin-1')

In [ ]:
data5.head()

In [ ]:
data5.drop(columns=["Unnamed: 0"], inplace=True)
data5.rename(columns={'Label': 'label', 'Body': 'text'}, inplace=True)

In [ ]:
# we have a problem on the first row. It starts with \n then identifies the subject then \n again. 
# If we were to split the text column with \n, we would have a problem.
# So when we are splitting, we will check if \ is the first character of the row
# if it is, we will find the next \n and split the text column with that
# if it is not, we will split the text column with \n

def custom_split(row):
    # Check if the value is NaN
    if pd.isna(row['text']):
        return np.nan, np.nan
    
    # Check if the first character is \
    if row['text'][0] == '\\':
        # Find the next \n after the first character
        next_backslash_index = row['text'].find('\n', 1)
        # Split the text column with the found \n
        return row['text'][1:next_backslash_index], row['text'][next_backslash_index+1:]
    else:
        # Split the text column with \n
        return row['text'].split('\n', 1) if '\n' in row['text'] else (row['text'], np.nan)

# Apply the function to your DataFrame
data5[['Subject', 'Text']] = data5.apply(custom_split, axis=1, result_type='expand')

In [ ]:
data5.head()

In [ ]:
data5.drop(columns=['text'], inplace=True)

In [ ]:
data5.head()

In [ ]:
data6 = pd.read_csv(PATH + "emails 2.csv", encoding='utf-8')

In [ ]:
data6.head()

In [ ]:
data6['message'][3]

In [ ]:
print(data6['message'][3])

In [ ]:
# https://www.kaggle.com/code/zichen/explore-enron/notebook 
# this notebook helped me to understand how to extract the content from the message column

import email 

def get_text_from_email(msg):
    parts = []
    for part in msg.walk():
        if part.get_content_type() == 'text/plain':
            parts.append( part.get_payload() )
    return ''.join(parts)

messages = list(map(email.message_from_string, data6['message']))
data6['content'] = list(map(get_text_from_email, messages))

keys = messages[0].keys()
for key in keys:
    data6[key] = [doc[key] for doc in messages]

In [ ]:
data6.head()

In [ ]:
data6.drop(columns=["Message-ID", "Date", "From", "To", "file", "Mime-Version", "Content-Type", "Content-Transfer-Encoding", "X-From", "X-To", "X-cc", "X-bcc", "X-Folder", "X-Origin", "X-FileName", "message"], inplace=True)

In [ ]:
data6.head()

In [ ]:
print(data1.columns)
print(data2.columns)
print(data3.columns)
print(data4.columns)
print(data5.columns)
print(data6.columns)

In [ ]:
# we will set all the names as label and text and if any, subject
data4.rename(columns={'spam': 'label', 'text': 'text', 'subject': 'subject'}, inplace=True)
data5.rename(columns={'Subject': 'subject', 'Text': 'text'}, inplace=True)
data6.rename(columns={'content': 'text', 'subject': 'subject'}, inplace=True)

In [ ]:
# we will now merge the first 5 datasets (train dataset)
# if there are no subject column, we will create one and set it as NaN

data = pd.concat([data1, data2, data3, data4, data5], ignore_index=True, sort=False)

In [ ]:
data.head()

In [ ]:
data.to_csv(PATH + "train.csv", index=False)

In [ ]:
data = pd.concat([data1, data2, data3, data4, data5, data6], ignore_index=True, sort=False)
data.to_csv(PATH + "zero_shot.csv", index=False)

In [ ]:
data6.to_csv(PATH + "seperate.csv", index=False)